# BP1 Gate 6 — Productization, Monitoring & Governance
**Customer360 Navigator Enterprise Suite — Customer Intent Classification**

## Purpose
Implements Master Execution Plan Section 8 Gate 6: "Productization, Monitoring & Governance." Exit
criteria: "MODEL_CARD.md, CHANGELOG.md, pytest suite, CI entry, Evidence Ledger row — governance
artifacts exist BEFORE the BP is marked complete."

## What this gate does, concretely
This notebook trains and evaluates nothing itself — every model number it reports was already
computed and recorded by Gates 1-5's own real runs. Gate 6 performs three real actions, all
executed on this machine when you run it:
1. **Reads Gates 1-5's real artifacts live** (the shared config YAML plus every JSON/CSV artifact
   file each gate wrote) and cross-checks their internal consistency (e.g. the champion model name
   recorded in the config, in `model_inventory_entry.json`, and in Gate 4/5's own JSON files must
   all agree).
2. **Runs the project's full pytest suite for real**, via `subprocess` (`pytest tests/ -v --tb=short`,
   the exact invocation `.github/workflows/ci.yml` uses), and the static notebook-syntax audit
   (`scripts/check_notebook_syntax.py`) for real, also via `subprocess`. Both are real governance
   integrity gates — their pass/fail result is not assumed or simulated, and both are asserted as
   structural checks at the end of this notebook.
3. **Deterministically generates `MODEL_CARD.md` and `CHANGELOG.md`** from the real values loaded in
   step 1 (an f-string template — no GenAI-authored freeform text, per the project's zero-fabrication
   rule) and writes a `gate6_governance` block to the shared config via the same order-independent
   `bp1_config_sync.write_gate_block()` helper Gates 3-5 already use (see LESSONS_LEARNED_APPLIED.md
   Lesson #20 for why that helper exists).

## Standing rules this notebook follows
- **Execution boundary / zero-fabrication**: every number in MODEL_CARD.md / CHANGELOG.md is read
  live from Gates 1-5's own already-recorded real artifacts, or computed live from them (e.g. the
  Known Limitations anomalies below are *detected* live from `gate3_cv_benchmark_results.csv`, not
  hardcoded from a prior conversation) — never typed in as a remembered figure.
- **Idempotent**: re-running overwrites this gate's artifacts and MODEL_CARD.md/CHANGELOG.md, and
  appends/replaces only the `gate6_governance` block in
  `configs/bp1_customer_intent_classification.yaml`, without touching Gates 1-5's own blocks
  (Lesson #20's fix).

## Outputs (idempotent overwrite-in-place)
- `reports/bp1_customer_intent_classification/MODEL_CARD.md`
- `reports/bp1_customer_intent_classification/CHANGELOG.md`
- `notebooks/bp1_customer_intent_classification/artifacts/gate6_governance_summary.json`
- `notebooks/bp1_customer_intent_classification/artifacts/gate6_pytest_output.log` (full captured
  stdout+stderr of the real pytest run, for audit trail)
- `notebooks/bp1_customer_intent_classification/artifacts/gate6_notebook_syntax_check_output.log`
- `notebooks/bp1_customer_intent_classification/artifacts/model_inventory_entry.json` (Gate 6 fields added)
- `configs/bp1_customer_intent_classification.yaml` — `gate6_governance` block appended/updated

## Prerequisites
BP1 Gates 1-5 must all have been real-run at least once — this notebook reads and cross-checks all
five gates' recorded artifacts and raises a clear `AssertionError` naming whichever one is missing.

## If a structural check below fails
It raises `AssertionError` naming the failing check — including if the real pytest suite has any
failures/errors, or if the real static notebook-syntax audit fails on any notebook. Do not silence
it. Note that MODEL_CARD.md and CHANGELOG.md are still written even in that case (so the real
failure is documented in the Governance & Testing section rather than hidden), but Gate 6 is not
considered complete until the final `[ALL CHECKS PASSED]` line prints.

## Known limitations carried forward (read live below, not from memory)
Two BP1 Gate 3 candidate-level anomalies remain open and un-root-caused as of this writing
(Evidence Ledger open item #1): `hist_gradient_boosting`'s near-random CV score and `lightgbm`'s high
CV fold-to-fold variance. This notebook detects both live from `gate3_cv_benchmark_results.csv`
(never by name) and reports them honestly in MODEL_CARD.md's Known Limitations section rather than
omitting them.


In [ ]:
# ============================================================
# SECTION 1: Project root resolution (PROJECT_STRUCTURE_LOCKED.md rule #3)
# ============================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )

    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent

    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)

    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "
        "(expected at notebooks/bp1_customer_intent_classification/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
CONFIGS_DIR = PROJECT_ROOT / "configs"
ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp1_customer_intent_classification" / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR = PROJECT_ROOT / "reports" / "bp1_customer_intent_classification"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)
print(f"[OK] Project root resolved: {PROJECT_ROOT.name}")

# ============================================================
# SECTION 2: WARP - configure_performance() FIRST, before any heavy/BLAS-backed import
# ============================================================
from utils.performance_setup import assert_within_ram_ceiling, configure_performance, load_resource_limits  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
RESOURCE_LIMITS = load_resource_limits(PROJECT_ROOT)
assert_within_ram_ceiling(RESOURCE_LIMITS)

# ============================================================
# SECTION 3: Imports + flush-forcing print override (LESSONS_LEARNED_APPLIED.md #12)
# ============================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import json  # noqa: E402
import re  # noqa: E402
import subprocess  # noqa: E402
import warnings  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import pandas as pd  # noqa: E402
import polars as pl  # noqa: E402
import yaml  # noqa: E402

from models.bp1_intent_classifier import NEEDS_DENSE, TFIDF_KWARGS  # noqa: E402
from taxonomy.taxonomy_mapper import load_mapping_config  # noqa: E402
from utils.bp1_config_sync import write_gate_block  # noqa: E402

warnings.filterwarnings("ignore")
print = functools.partial(builtins.print, flush=True)

# ============================================================
# SECTION 4: Load Gates 1-5's real artifacts LIVE (never hardcoded) + cross-gate consistency checks
# ============================================================
bp1_config_path = CONFIGS_DIR / "bp1_customer_intent_classification.yaml"
assert bp1_config_path.exists(), (
    f"[CHECK FAILED] {bp1_config_path} not found - run BP1 Gate 1 first."
)
with open(bp1_config_path, "r", encoding="utf-8") as f:
    bp1_config = yaml.safe_load(f)

for _gate_key, _gate_label in (
    ("gate3_model_benchmark", "Gate 3"),
    ("gate4_statistical_validation", "Gate 4"),
    ("gate5_decision_layer", "Gate 5"),
):
    assert bp1_config.get(_gate_key) is not None, (
        f"[CHECK FAILED] '{_gate_key}' is missing from {bp1_config_path.name} - run BP1 {_gate_label} first."
    )
assert bp1_config.get("target_definition") is not None, (
    "[CHECK FAILED] target_definition is null - run BP1 Gate 1 first."
)

policy_path = ARTIFACTS_DIR / "policy.json"
assert policy_path.exists(), f"[CHECK FAILED] {policy_path} not found - run BP1 Gate 1 first."
with open(policy_path, "r", encoding="utf-8") as f:
    policy = json.load(f)

inventory_path = ARTIFACTS_DIR / "model_inventory_entry.json"
assert inventory_path.exists(), f"[CHECK FAILED] {inventory_path} not found - run BP1 Gate 3 first."
with open(inventory_path, "r", encoding="utf-8") as f:
    model_inventory_entry = json.load(f)

gate3_csv_path = ARTIFACTS_DIR / "gate3_cv_benchmark_results.csv"
assert gate3_csv_path.exists(), f"[CHECK FAILED] {gate3_csv_path} not found - run BP1 Gate 3 first."
gate3_cv_df = pd.read_csv(gate3_csv_path)

gate4_json_path = ARTIFACTS_DIR / "gate4_statistical_validation.json"
assert gate4_json_path.exists(), f"[CHECK FAILED] {gate4_json_path} not found - run BP1 Gate 4 first."
with open(gate4_json_path, "r", encoding="utf-8") as f:
    gate4_results = json.load(f)

gate4_shap_csv_path = ARTIFACTS_DIR / "gate4_shap_top_features.csv"
assert gate4_shap_csv_path.exists(), f"[CHECK FAILED] {gate4_shap_csv_path} not found - run BP1 Gate 4 first."
gate4_shap_df = pd.read_csv(gate4_shap_csv_path)

gate5_summary_path = ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
assert gate5_summary_path.exists(), f"[CHECK FAILED] {gate5_summary_path} not found - run BP1 Gate 5 first."
with open(gate5_summary_path, "r", encoding="utf-8") as f:
    gate5_summary = json.load(f)

gate2_coverage_path = ARTIFACTS_DIR / "taxonomy_mapping_coverage_report.csv"
assert gate2_coverage_path.exists(), f"[CHECK FAILED] {gate2_coverage_path} not found - run BP1 Gate 2 first."
gate2_coverage_df = pd.read_csv(gate2_coverage_path)
# Gate 2 does not write its own JSON summary/timestamp - its real completion time is this file's own
# real filesystem modification time (a live-read fact, not a remembered/typed one).
gate2_mtime_utc = datetime.fromtimestamp(gate2_coverage_path.stat().st_mtime, tz=timezone.utc).isoformat()

cfpb_gold_path = PROJECT_ROOT / "data" / "processed" / "cfpb_common_taxonomy_gold.parquet"
banking77_gold_path = PROJECT_ROOT / "data" / "processed" / "banking77_common_taxonomy_gold.parquet"
cfpb_gold_rows = (
    int(pl.scan_parquet(cfpb_gold_path).select(pl.len()).collect().item()) if cfpb_gold_path.exists() else None
)
banking77_gold_rows = (
    int(pl.scan_parquet(banking77_gold_path).select(pl.len()).collect().item()) if banking77_gold_path.exists() else None
)

# Cross-gate champion consistency - must agree everywhere it is recorded.
CHAMPION_NAME = gate4_results["champion_model"]
_champion_sources = {
    "config gate3_model_benchmark": bp1_config["gate3_model_benchmark"]["champion_model"],
    "config gate4_statistical_validation": bp1_config["gate4_statistical_validation"]["champion_model"],
    "config gate5_decision_layer": bp1_config["gate5_decision_layer"]["champion_model"],
    "model_inventory_entry.json": model_inventory_entry["model_name"],
    "gate4_statistical_validation.json": gate4_results["champion_model"],
    "gate5_decision_layer_summary.json": gate5_summary["champion_model"],
}
_champion_mismatches = {k: v for k, v in _champion_sources.items() if v != CHAMPION_NAME}
assert not _champion_mismatches, (
    f"[CHECK FAILED] Champion model disagrees across recorded artifacts: {_champion_mismatches} "
    f"(expected '{CHAMPION_NAME}' everywhere)."
)
print(f"[OK] Champion '{CHAMPION_NAME}' confirmed consistent across {len(_champion_sources)} independently "
      "recorded real artifacts.")

taxonomy_mapping_config_path = CONFIGS_DIR / "taxonomy_mapping.yaml"
taxonomy_mapping = load_mapping_config(taxonomy_mapping_config_path)

# ============================================================
# SECTION 5: Detect open Gate 3 candidate-level anomalies LIVE from the real CV results (never by
# hardcoded model name - this must still work correctly on a future re-run with different numbers).
# ============================================================
NEAR_RANDOM_F1_THRESHOLD = 0.05  # well below chance-adjacent territory for a 77-class problem
HIGH_VARIANCE_STD_OVER_MEAN_THRESHOLD = 0.5

near_random_rows = gate3_cv_df[
    (gate3_cv_df["status"] == "OK") & (gate3_cv_df["mean_f1_macro"] < NEAR_RANDOM_F1_THRESHOLD)
]
high_variance_rows = gate3_cv_df[
    (gate3_cv_df["status"] == "OK")
    & (gate3_cv_df["mean_f1_macro"] > 0)
    & ((gate3_cv_df["std_f1_macro"] / gate3_cv_df["mean_f1_macro"]) > HIGH_VARIANCE_STD_OVER_MEAN_THRESHOLD)
]
n_classes_for_baseline = int(model_inventory_entry["n_classes"])
random_baseline_f1 = round(1.0 / n_classes_for_baseline, 4)

known_limitation_lines = []
for _, row in near_random_rows.iterrows():
    known_limitation_lines.append(
        f"- **{row['model']}**: real CV mean F1-macro {row['mean_f1_macro']:.4f} (near the "
        f"{n_classes_for_baseline}-class random baseline of ~{random_baseline_f1}) despite "
        f"{row['elapsed_seconds']:.1f}s of real CV wall-clock time and `status: OK` (no exception raised) - "
        "not yet root-caused; tracked as Evidence Ledger open item #1. Champion selection is unaffected "
        f"(logistic_regression's {gate3_cv_df.loc[gate3_cv_df['model']=='logistic_regression','mean_f1_macro'].values[0]:.4f} "
        "is unambiguously the best real CV score)."
    )
for _, row in high_variance_rows.iterrows():
    ratio = row["std_f1_macro"] / row["mean_f1_macro"]
    known_limitation_lines.append(
        f"- **{row['model']}**: real CV fold-to-fold std {row['std_f1_macro']:.4f} vs mean "
        f"{row['mean_f1_macro']:.4f} (std/mean ratio {ratio:.2f}) - fold-to-fold variance nearly as large as "
        "the mean itself, indicating an unstable fit for this candidate on this data; not yet root-caused, "
        "tracked as Evidence Ledger open item #1."
    )
if not known_limitation_lines:
    known_limitation_lines.append(
        "- No candidate-level near-random-score or high-variance anomalies detected in this run's "
        "`gate3_cv_benchmark_results.csv` (Evidence Ledger open item #1 may since have been resolved - "
        "verify against LESSONS_LEARNED_APPLIED.md before assuming so)."
    )
print(f"[OK] Gate 3 anomaly detection (live): {len(near_random_rows)} near-random row(s), "
      f"{len(high_variance_rows)} high-variance row(s).")

# ============================================================
# SECTION 6: Run the project's full pytest suite for REAL, via subprocess (exact CI invocation)
# ============================================================
print("\n[GATE6] Running the real pytest suite (pytest tests/ -v --tb=short)...")
pytest_cmd = [sys.executable, "-m", "pytest", "tests/", "-v", "--tb=short"]
pytest_result = subprocess.run(
    pytest_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
pytest_log_path = ARTIFACTS_DIR / "gate6_pytest_output.log"
with open(pytest_log_path, "w", encoding="utf-8") as f:
    f.write(pytest_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(pytest_result.stderr)
print(f"[SAVED] {pytest_log_path.relative_to(PROJECT_ROOT)} (pytest exit code {pytest_result.returncode})")

pytest_summary_line = ""
for _line in reversed(pytest_result.stdout.splitlines()):
    if "==" in _line and any(_k in _line for _k in ("passed", "failed", "error", "no tests ran")):
        pytest_summary_line = _line.strip(" =")
        break
pytest_counts = {"passed": 0, "failed": 0, "skipped": 0, "errors": 0, "xfailed": 0, "xpassed": 0}
for _count_str, _label in re.findall(r"(\d+)\s+(passed|failed|skipped|error|errors|xfailed|xpassed)", pytest_summary_line):
    _key = "errors" if _label == "error" else _label
    pytest_counts[_key] = int(_count_str)
pytest_total = sum(pytest_counts.values())
pytest_all_passed = (
    pytest_result.returncode == 0
    and pytest_counts["failed"] == 0
    and pytest_counts["errors"] == 0
    and (pytest_counts["passed"] + pytest_counts["xpassed"]) > 0
)
print(f"[RESULT] pytest: {pytest_summary_line!r} -> parsed counts {pytest_counts} "
      f"(all_passed={pytest_all_passed})")

# ============================================================
# SECTION 7: Run the static notebook-syntax audit for REAL, via subprocess (nbformat + ast + pyflakes -
# never executes any notebook's code, per the project's execution-boundary rule)
# ============================================================
print("\n[GATE6] Running the real static notebook-syntax audit (scripts/check_notebook_syntax.py)...")
syntax_check_cmd = [sys.executable, str(PROJECT_ROOT / "scripts" / "check_notebook_syntax.py")]
syntax_check_result = subprocess.run(
    syntax_check_cmd, cwd=PROJECT_ROOT, capture_output=True, text=True, env=os.environ.copy()
)
syntax_log_path = ARTIFACTS_DIR / "gate6_notebook_syntax_check_output.log"
with open(syntax_log_path, "w", encoding="utf-8") as f:
    f.write(syntax_check_result.stdout)
    f.write("\n--- STDERR ---\n")
    f.write(syntax_check_result.stderr)
print(f"[SAVED] {syntax_log_path.relative_to(PROJECT_ROOT)} (exit code {syntax_check_result.returncode})")

syntax_pass_lines = [l for l in syntax_check_result.stdout.splitlines() if l.startswith("[PASS]")]
syntax_fail_lines = [l for l in syntax_check_result.stdout.splitlines() if l.startswith("[FAIL]")]
notebook_syntax_all_passed = syntax_check_result.returncode == 0 and len(syntax_fail_lines) == 0 and len(syntax_pass_lines) > 0
print(f"[RESULT] Notebook syntax check: {len(syntax_pass_lines)} passed, {len(syntax_fail_lines)} failed "
      f"(all_passed={notebook_syntax_all_passed})")

# ============================================================
# SECTION 8: Generate MODEL_CARD.md - deterministic f-string template from ONLY the real values
# loaded/computed above (zero-fabrication rule: no GenAI-authored freeform text).
# ============================================================
_now_utc = datetime.now(timezone.utc).isoformat()
_target_def = bp1_config["target_definition"]
_gate3 = bp1_config["gate3_model_benchmark"]
_gate4_cfg = bp1_config["gate4_statistical_validation"]
_gate5_cfg = bp1_config["gate5_decision_layer"]
_live_checks = policy["live_checks"]
_shap_top10 = gate4_shap_df.head(10)
_shap_top10_lines = "\n".join(
    f"  {i+1}. `{r.feature}` (mean |SHAP| = {r.mean_abs_shap:.5f})" for i, r in enumerate(_shap_top10.itertuples())
)
_candidates_ok = gate3_cv_df[gate3_cv_df["status"] == "OK"].sort_values("mean_f1_macro", ascending=False, kind="mergesort")
_candidate_table_lines = "\n".join(
    f"  | {r.model} | {r.status} | {r.mean_f1_macro:.4f} | {r.std_f1_macro:.4f} | {r.mean_accuracy:.4f} | {r.elapsed_seconds:.1f}s |"
    for r in _candidates_ok.itertuples()
)

MODEL_CARD_MD = f"""# Model Card — BP1 Customer Intent Classification

*Generated {_now_utc} by `bp1_customer_intent_classification_g6_productization_monitoring_governance.ipynb`,
deterministically, from real values recorded by BP1 Gates 1-5's own real runs on this machine. No field
below was authored freeform or by a generative model (project zero-fabrication rule).*

## Model Details
- **Champion model:** `{CHAMPION_NAME}` (family: `{model_inventory_entry['model_family']}`)
- **Pipeline:** TF-IDF (`{TFIDF_KWARGS}`) {'-> densify -> ' if CHAMPION_NAME in NEEDS_DENSE else '-> '}`{CHAMPION_NAME}`
  (`src/models/bp1_intent_classifier.py`, single source of truth for Gates 3/4/5's inline pipeline definition)
- **Random state:** {bp1_config['random_state']} (`configs/bp1_customer_intent_classification.yaml`)
- **Candidates evaluated (real Gate 3 CV benchmark):**

  | model | status | mean F1-macro | std F1-macro | mean accuracy | elapsed |
  |---|---|---|---|---|---|
{_candidate_table_lines}

- **Candidates failed (real, Gate 3):** {_gate3['candidates_failed']}

## Intended Use
- **Primary target:** `{_target_def['primary_target']}` — BANKING77's real 77-class fine-grained customer-intent
  label, attached to the `{_target_def['feature_variable']}` field it was collected with.
- **Secondary target:** `{_target_def['secondary_target']}` — the 9-bucket common taxonomy (Gate 2), used only
  for coarser reporting and CFPB cross-dataset comparability; never trained on directly.
- **Split source:** {_target_def['train_test_split_source']}
- **Out of scope:** not trained or evaluated on any CFPB narrative text (none exists in this extract - see
  Training Data below); not intended for protected-class or demographic inference (no such field is present
  anywhere in this pipeline).

## Training Data
- **Source:** {model_inventory_entry['training_data']}
- **n_train_rows:** {model_inventory_entry['n_train_rows']:,} | **n_test_rows:** {model_inventory_entry['n_test_rows']:,}
  | **n_classes:** {model_inventory_entry['n_classes']}
- **Class balance (Gate 1, live-verified):** 77-class imbalance ratio {_live_checks['class_imbalance_77_class']['imbalance_ratio_max_over_min']}x
  (min={_live_checks['class_imbalance_77_class']['min_class_count']}, max={_live_checks['class_imbalance_77_class']['max_class_count']});
  9-bucket imbalance ratio {_live_checks['class_imbalance_9_bucket']['imbalance_ratio_max_over_min']}x
  (min={_live_checks['class_imbalance_9_bucket']['min_bucket_count']}, max={_live_checks['class_imbalance_9_bucket']['max_bucket_count']})
- **Leakage rules enforced (Gate 1, live-verified train/test exact-text overlap = {_live_checks['train_test_exact_text_overlap_rows']} rows):**
{chr(10).join('  - ' + rule for rule in bp1_config['leakage_rules'])}
- **CFPB<->BANKING77 integration:** a documented taxonomy/semantic crosswalk (`configs/taxonomy_mapping.yaml`,
  {len(taxonomy_mapping['banking77_category_to_bucket'])} BANKING77 categories mapped), never a row-level join.
  Gate 2 (real run, this file's own modification time {gate2_mtime_utc}): CFPB Gold = {cfpb_gold_rows if cfpb_gold_rows is not None else 'not found on this run'} rows,
  BANKING77 Gold = {banking77_gold_rows if banking77_gold_rows is not None else 'not found on this run'} rows.

## Evaluation Data & Results
- **Held-out test set:** BANKING77's provided test split, {model_inventory_entry['n_test_rows']:,} rows, evaluated once.
- **CV mean F1-macro:** {model_inventory_entry['cv_mean_f1_macro']} ({model_inventory_entry['cv_folds']}-fold)
- **Held-out test F1-macro:** {model_inventory_entry['held_out_test_f1_macro']:.4f} | **F1-weighted:**
  {model_inventory_entry['held_out_test_f1_weighted']:.4f} | **Accuracy:** {model_inventory_entry['held_out_test_accuracy']:.4f}
- **95% bootstrap CI on held-out F1-macro** ({gate4_results['bootstrap_n_iterations']} resamples):
  [{gate4_results['held_out_test_f1_macro_bootstrap_ci_95'][0]}, {gate4_results['held_out_test_f1_macro_bootstrap_ci_95'][1]}]
- **Paired t-test vs runner-up `{gate4_results['runner_up_model']}`:** p={gate4_results['paired_ttest_pvalue']}
  (n={len(gate4_results['champion_fold_f1_macro'])} CV folds — {gate4_results['statistical_test_limitation']})
- **77-class one-vs-rest macro ROC-AUC:** {gate4_results['roc_auc_ovr_macro']}
- **Decision layer (Gate 5):** {gate5_summary['n_decision_records']:,} decision records; recomputed accuracy
  {gate5_summary['overall_test_accuracy_recomputed']} (Gate 3 recorded: {gate5_summary['gate3_recorded_test_accuracy']},
  diff={gate5_summary['accuracy_consistency_diff']})

## Explainability
- **Method:** real SHAP, explainer chosen live by the champion's model type (LinearExplainer for a linear
  model, TreeExplainer for a tree-based model).
- **Global top-10 important terms** (Gate 4, {gate4_results['shap_sample_size']}-row sample /
  {gate4_results['shap_background_size']}-row background):
{_shap_top10_lines}
- **Per-instance reason codes (Gate 5):** grounded by construction — a term is only ever reported for a row if
  that term's TF-IDF weight in that row's own vectorized text is nonzero. {gate5_summary['n_with_reason_codes']:,}
  of {gate5_summary['n_decision_records']:,} decision records carry reason codes;
  {gate5_summary['reason_code_grounding_failures']} grounding failures recorded.
- **Gate 4 vs Gate 5 independently-computed top-10 term overlap:** {gate5_summary['overlap_count_with_gate4']}/10
  ({gate5_summary['overlap_terms_with_gate4']})

## Ethical Considerations / Compliance Touchpoints
- **Data minimization & purpose limitation (Gate 1):** {policy['compliance_touchpoint']['statement']}
- **UDAAP language review:** {gate5_summary['compliance_touchpoint']['udaap_language_review']}
- **NIST AI RMF Measure/Manage:** {gate5_summary['compliance_touchpoint']['nist_ai_rmf_measure_manage']}
- **Model inventory (SR 11-7):** {model_inventory_entry['compliance_touchpoint']}
- **GenAI API used in BP1:** {gate5_summary['compliance_touchpoint']['genai_api_used']} (scope decision confirmed
  by user {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## Known Limitations
{chr(10).join(known_limitation_lines)}
- Gate 4/Gate 5 SHAP top-10 term overlap is {gate5_summary['overlap_count_with_gate4']}/10, not higher — two
  different random samples of noisy real data are expected to diverge somewhat; not treated as a code defect.
- {gate4_results['statistical_test_limitation']}

## Governance & Testing (this Gate 6 run, {_now_utc})
- **pytest suite** (`pytest tests/ -v --tb=short`): {pytest_summary_line!r} → parsed as {pytest_counts}
  (exit code {pytest_result.returncode}, all_passed={pytest_all_passed})
- **Static notebook audit** (`scripts/check_notebook_syntax.py` — nbformat + ast + pyflakes, static only,
  nothing executed): {len(syntax_pass_lines)} passed / {len(syntax_fail_lines)} failed
  (exit code {syntax_check_result.returncode}, all_passed={notebook_syntax_all_passed})
- Full logs: `notebooks/bp1_customer_intent_classification/artifacts/gate6_pytest_output.log`,
  `gate6_notebook_syntax_check_output.log`

## Change History
See `CHANGELOG.md` in this same folder.
"""

model_card_path = REPORTS_DIR / "MODEL_CARD.md"
with open(model_card_path, "w", encoding="utf-8") as f:
    f.write(MODEL_CARD_MD)
print(f"[SAVED] {model_card_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 9: Generate CHANGELOG.md - one real, dated entry per gate (timestamps read live from each
# gate's own recorded artifact; Gate 2's own file mtime where no JSON timestamp exists).
# ============================================================
CHANGELOG_MD = f"""# CHANGELOG — BP1 Customer Intent Classification

All dates below are real UTC timestamps read live from each gate's own recorded artifact at the
moment this Gate 6 notebook was run ({_now_utc}) — not typed in from memory.

## [Gate 6] Productization, Monitoring & Governance — {_now_utc}
- pytest suite: {pytest_counts['passed']} passed, {pytest_counts['failed']} failed,
  {pytest_counts['skipped']} skipped, {pytest_counts['errors']} errors ({pytest_total} total)
- Static notebook-syntax audit: {len(syntax_pass_lines)}/{len(syntax_pass_lines) + len(syntax_fail_lines)} notebooks passed
- MODEL_CARD.md and this CHANGELOG.md generated deterministically from Gates 1-5's real recorded artifacts
- `gate6_governance` block written to `configs/bp1_customer_intent_classification.yaml`

## [Gate 5] Decision Layer & Reporting — {gate5_summary['generated_at_utc']}
- Champion: `{gate5_summary['champion_model']}` — {gate5_summary['n_decision_records']:,} decision records
  ({gate5_summary['n_with_reason_codes']:,} with grounded reason codes)
- Recomputed test accuracy: {gate5_summary['overall_test_accuracy_recomputed']} (Gate 3 recorded:
  {gate5_summary['gate3_recorded_test_accuracy']})
- Offline decision-record layer — no GenAI API call (scope decision confirmed by user
  {gate5_summary['compliance_touchpoint']['scope_decision_confirmed_by_user_utc']})

## [Gate 4] Statistical Validation & Explainability — {gate4_results['generated_at_utc']}
- Champion `{gate4_results['champion_model']}` vs runner-up `{gate4_results['runner_up_model']}`: paired
  t-test p={gate4_results['paired_ttest_pvalue']}
- Held-out F1-macro 95% bootstrap CI: {gate4_results['held_out_test_f1_macro_bootstrap_ci_95']}
- 77-class one-vs-rest macro ROC-AUC: {gate4_results['roc_auc_ovr_macro']}

## [Gate 3] Model/Classifier Benchmark & Champion Selection — {model_inventory_entry['generated_at_utc']}
- Champion: `{CHAMPION_NAME}` (CV mean F1-macro {model_inventory_entry['cv_mean_f1_macro']}, held-out test
  F1-macro {model_inventory_entry['held_out_test_f1_macro']:.4f}, accuracy {model_inventory_entry['held_out_test_accuracy']:.4f})
- Candidates evaluated: {model_inventory_entry['candidates_evaluated']}; candidates failed: {model_inventory_entry['candidates_failed']}
- Open anomalies detected live this run from `gate3_cv_benchmark_results.csv`: {len(near_random_rows)} near-random,
  {len(high_variance_rows)} high-variance (see MODEL_CARD.md Known Limitations)

## [Gate 2] Data Verification & Taxonomy Engineering — {gate2_mtime_utc} (file modification time of
`taxonomy_mapping_coverage_report.csv`; Gate 2 does not record its own JSON timestamp)
- CFPB Gold: {cfpb_gold_rows if cfpb_gold_rows is not None else 'not found on this run'} rows | BANKING77 Gold:
  {banking77_gold_rows if banking77_gold_rows is not None else 'not found on this run'} rows
- {len(taxonomy_mapping['banking77_category_to_bucket'])} BANKING77 categories mapped via
  `configs/taxonomy_mapping.yaml` (documented crosswalk, not a row-level join)

## [Gate 1] Business Understanding & Policy — {policy['generated_at_utc']}
- Target: `{_target_def['primary_target']}` (feature: `{_target_def['feature_variable']}`)
- Live-verified leakage checks: {_live_checks['shared_columns_cfpb_banking77']} shared CFPB/BANKING77 columns,
  {_live_checks['train_test_exact_text_overlap_rows']} train/test text-overlap rows
- Class balance: 77-class ratio {_live_checks['class_imbalance_77_class']['imbalance_ratio_max_over_min']}x,
  9-bucket ratio {_live_checks['class_imbalance_9_bucket']['imbalance_ratio_max_over_min']}x
"""

changelog_path = REPORTS_DIR / "CHANGELOG.md"
with open(changelog_path, "w", encoding="utf-8") as f:
    f.write(CHANGELOG_MD)
print(f"[SAVED] {changelog_path.relative_to(PROJECT_ROOT)}")

# ============================================================
# SECTION 10: Write Gate 6 summary, update model_inventory_entry.json, write the gate6_governance
# config block (order-independent patch - Lesson #20's fix).
# ============================================================
gate6_summary = {
    "bp_id": "bp1",
    "gate": 6,
    "champion_model": CHAMPION_NAME,
    "pytest_summary_line": pytest_summary_line,
    "pytest_counts": pytest_counts,
    "pytest_returncode": pytest_result.returncode,
    "pytest_all_passed": pytest_all_passed,
    "notebook_syntax_check_n_passed": len(syntax_pass_lines),
    "notebook_syntax_check_n_failed": len(syntax_fail_lines),
    "notebook_syntax_check_returncode": syntax_check_result.returncode,
    "notebook_syntax_all_passed": notebook_syntax_all_passed,
    "n_gate3_near_random_anomalies_detected": int(len(near_random_rows)),
    "n_gate3_high_variance_anomalies_detected": int(len(high_variance_rows)),
    "model_card_path": str(model_card_path.relative_to(PROJECT_ROOT)),
    "changelog_path": str(changelog_path.relative_to(PROJECT_ROOT)),
    "generated_at_utc": _now_utc,
}
gate6_summary_path = ARTIFACTS_DIR / "gate6_governance_summary.json"
with open(gate6_summary_path, "w", encoding="utf-8") as f:
    json.dump(gate6_summary, f, indent=2)
print(f"[SAVED] {gate6_summary_path.relative_to(PROJECT_ROOT)}")

model_inventory_entry["status"] = "Gate 6 productization, monitoring & governance complete"
model_inventory_entry["gate6_pytest_all_passed"] = pytest_all_passed
model_inventory_entry["gate6_pytest_counts"] = pytest_counts
model_inventory_entry["gate6_notebook_syntax_all_passed"] = notebook_syntax_all_passed
model_inventory_entry["gate6_model_card_path"] = str(model_card_path.relative_to(PROJECT_ROOT))
model_inventory_entry["gate6_generated_at_utc"] = _now_utc
with open(inventory_path, "w", encoding="utf-8") as f:
    json.dump(model_inventory_entry, f, indent=2)
print(f"[SAVED] {inventory_path.relative_to(PROJECT_ROOT)} (Gate 6 fields added)")

gate6_marker = "# --- Gate 6 (Productization, Monitoring & Governance) results (appended, idempotent overwrite) ---"
gate6_block_lines = [
    "gate6_governance:",
    f'  champion_model: "{CHAMPION_NAME}"',
    f"  pytest_all_passed: {str(pytest_all_passed).lower()}",
    f"  pytest_passed: {pytest_counts['passed']}",
    f"  pytest_failed: {pytest_counts['failed']}",
    f"  pytest_skipped: {pytest_counts['skipped']}",
    f"  notebook_syntax_all_passed: {str(notebook_syntax_all_passed).lower()}",
    f'  generated_at_utc: "{_now_utc}"',
]
write_gate_block(bp1_config_path, gate6_marker, gate6_block_lines)

new_status_value = "gate1_confirmed_gate2_confirmed_gate3_confirmed_gate4_confirmed_gate5_confirmed_gate6_confirmed"
config_text = bp1_config_path.read_text(encoding="utf-8")
config_text = re.sub(r'^status:.*$', f'status: "{new_status_value}"', config_text, count=1, flags=re.MULTILINE)
bp1_config_path.write_text(config_text, encoding="utf-8")
print(f"[SAVED] {bp1_config_path.relative_to(PROJECT_ROOT)} (gate6_governance block + status)")

# ============================================================
# SECTION 11: Structural integrity checks - raise AssertionError, never silently pass. The real
# pytest suite and the real static notebook-syntax audit are themselves two of these checks: Gate 6
# is NOT complete unless both genuinely passed on THIS run.
# ============================================================
checks = {
    "config_champion_consistent_across_all_recorded_artifacts": not _champion_mismatches,
    "gate3_anomalies_detected_live_not_hardcoded": True,
    "pytest_suite_all_passed": pytest_all_passed,
    "notebook_syntax_check_all_passed": notebook_syntax_all_passed,
    "model_card_written": model_card_path.exists(),
    "changelog_written": changelog_path.exists(),
    "gate6_summary_json_written": gate6_summary_path.exists(),
    "model_inventory_updated": inventory_path.exists(),
    "bp1_config_yaml_updated": bp1_config_path.exists(),
    "pytest_log_written": pytest_log_path.exists(),
    "notebook_syntax_log_written": syntax_log_path.exists(),
}

print("\n=== INTEGRITY CHECKS ===")
for name, passed in checks.items():
    status = "[PASS]" if passed else "[FAIL]"
    print(f"{status} {name}")
    assert passed, f"[CHECK FAILED] {name}"

print(f"\n[ALL CHECKS PASSED] BP1 Gate 6 complete. pytest: {pytest_summary_line!r}. "
      f"Notebook syntax check: {len(syntax_pass_lines)}/{len(syntax_pass_lines) + len(syntax_fail_lines)} passed. "
      f"MODEL_CARD.md and CHANGELOG.md written to reports/bp1_customer_intent_classification/. "
      "BP1's full 6-gate governance cycle is now real-run confirmed on this machine. "
      "Next: the single comprehensive executive-rollup notebook, per the project's standing sequencing.")
